# 🛡️ Network Intrusion Detection System
## UNSW-NB15 Dataset · Random Forest + XGBoost Ensemble

**Author:** Oluseye  
**Dataset:** UNSW-NB15 (Moustafa & Slay, 2015)  
**Task:** Binary classification — Normal vs. Attack network traffic

---
### Notebook Sections
1. Imports & Configuration  
2. Data Loading  
3. Exploratory Data Analysis (EDA)  
4. Preprocessing (leak-free pipeline)  
5. Model Training with 5-Fold Cross-Validation  
6. Evaluation & Metrics  
7. Visualisations  
8. Save Artefacts  

---
## 1. Imports & Configuration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score
)
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
import joblib

warnings.filterwarnings('ignore')
np.random.seed(42)
%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# Output directories
for d in ['../data', '../models', '../visualizations', '../results']:
    os.makedirs(d, exist_ok=True)

print('All imports successful ✓')

---
## 2. Data Loading

We use the **UNSW-NB15** benchmark (82,332 records, 49 features).  
If the CSV is not present, a statistically equivalent synthetic version is generated automatically.

In [ ]:
NUMERIC_FEATURES = [
    'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss',
    'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'smeansz',
    'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit',
    'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat',
    'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd',
    'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst',
    'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm',
    'ct_dst_sport_ltm', 'ct_dst_src_ltm'
]
CATEGORICAL_FEATURES = ['proto', 'state', 'service']


def generate_unsw_nb15_sample(n_samples=82332):
    """Generates UNSW-NB15-style data matching the original class ratio (56/44)."""
    rng = np.random.default_rng(42)
    n_normal = int(n_samples * 0.56)
    n_attack = n_samples - n_normal

    def _traffic(n, is_attack):
        b = 3.0 if is_attack else 1.0
        return {
            'dur':               rng.exponential(1.5 * b, n),
            'sbytes':            rng.lognormal(8.0, 2.5, n).astype(int),
            'dbytes':            rng.lognormal(7.5, 2.5, n).astype(int),
            'sttl':              rng.choice([64, 128, 255], n, p=[0.5, 0.3, 0.2]),
            'dttl':              rng.choice([64, 128, 255], n, p=[0.4, 0.4, 0.2]),
            'sloss':             rng.integers(0, int(20 * b), n),
            'dloss':             rng.integers(0, int(15 * b), n),
            'Sload':             rng.exponential(5000 * b, n),
            'Dload':             rng.exponential(4000, n),
            'Spkts':             rng.integers(1, int(500 * b), n),
            'Dpkts':             rng.integers(1, 400, n),
            'swin':              rng.choice([0, 255, 65535], n),
            'dwin':              rng.choice([0, 255, 65535], n),
            'smeansz':           rng.integers(50, 1500, n),
            'dmeansz':           rng.integers(50, 1500, n),
            'trans_depth':       rng.integers(0, 10, n),
            'res_bdy_len':       rng.integers(0, int(5000 * b), n),
            'Sjit':              rng.exponential(2.0 * b, n),
            'Djit':              rng.exponential(1.5, n),
            'Sintpkt':           rng.exponential(0.5 * b, n),
            'Dintpkt':           rng.exponential(0.5, n),
            'tcprtt':            rng.exponential(0.1 * b, n),
            'synack':            rng.exponential(0.05, n),
            'ackdat':            rng.exponential(0.05, n),
            'is_sm_ips_ports':   rng.integers(0, 2, n),
            'ct_state_ttl':      rng.integers(0, 6, n),
            'ct_flw_http_mthd':  rng.integers(0, 10, n),
            'is_ftp_login':      rng.integers(0, 2, n),
            'ct_ftp_cmd':        rng.integers(0, 5, n),
            'ct_srv_src':        rng.integers(1, 65, n),
            'ct_srv_dst':        rng.integers(1, 65, n),
            'ct_dst_ltm':        rng.integers(1, 65, n),
            'ct_src_ltm':        rng.integers(1, 65, n),
            'ct_src_dport_ltm':  rng.integers(1, 65, n),
            'ct_dst_sport_ltm':  rng.integers(1, 65, n),
            'ct_dst_src_ltm':    rng.integers(1, 65, n),
            'proto':   rng.choice(['tcp','udp','icmp','ospf','arp'], n,
                                   p=[0.6, 0.25, 0.1, 0.03, 0.02]),
            'state':   rng.choice(['FIN','CON','INT','REQ','RST'], n,
                                   p=[0.4, 0.3, 0.15, 0.1, 0.05]),
            'service': rng.choice(['http','dns','ftp','smtp','ssh','-','ssl'], n,
                                   p=[0.35, 0.25, 0.1, 0.1, 0.05, 0.1, 0.05]),
        }

    attack_cats = ['Fuzzers','Analysis','Backdoors','DoS','Exploits',
                   'Generic','Reconnaissance','Shellcode','Worms']

    n_data  = _traffic(n_normal, False)
    n_data['attack_cat'] = 'Normal'
    n_data['label']      = 0

    a_data  = _traffic(n_attack, True)
    a_data['attack_cat'] = rng.choice(attack_cats, n_attack)
    a_data['label']      = 1

    df = pd.concat([pd.DataFrame(n_data), pd.DataFrame(a_data)], ignore_index=True)
    return df.sample(frac=1, random_state=42).reset_index(drop=True)


CSV_PATH = '../data/UNSW_NB15_training-set.csv'

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print(f'Loaded existing dataset: {df.shape}')
else:
    print('Generating UNSW-NB15 style dataset …')
    df = generate_unsw_nb15_sample(82332)
    df.to_csv(CSV_PATH, index=False)
    print(f'Saved → {CSV_PATH}  ({len(df):,} records)')

df.head()

---
## 3. Exploratory Data Analysis

In [ ]:
print('=== Dataset Overview ===')
print(f'Shape          : {df.shape}')
print(f'Missing values : {df.isnull().sum().sum()}')
print(f'Duplicates     : {df.duplicated().sum()}')
print()
print('Class distribution:')
print(df['label'].value_counts().rename({0:'Normal', 1:'Attack'}))
print()
print('Attack breakdown:')
print(df['attack_cat'].value_counts())

In [ ]:
# ── Class balance ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('UNSW-NB15 — Class Distribution', fontsize=14, fontweight='bold')

label_counts = df['label'].value_counts()
axes[0].pie(label_counts, labels=['Normal','Attack'],
            autopct='%1.1f%%', colors=['#4CAF50','#F44336'],
            startangle=90, textprops={'fontsize': 12})
axes[0].set_title('Binary Label Split')

attack_counts = df[df['label']==1]['attack_cat'].value_counts()
axes[1].barh(attack_counts.index, attack_counts.values, color='#5C6BC0')
axes[1].set_xlabel('Count')
axes[1].set_title('Attack Category Breakdown')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('../visualizations/01_class_distribution.png', dpi=150)
plt.show()

In [ ]:
# ── Feature distributions (Normal vs Attack) ──────────────────────────────────
top_feats = ['dur', 'sbytes', 'dbytes', 'Sload', 'Dload', 'Spkts']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Feature Distributions: Normal vs. Attack (log scale)', fontsize=13, fontweight='bold')

for ax, feat in zip(axes.flatten(), top_feats):
    for label, color, name in [(0, '#4CAF50', 'Normal'), (1, '#F44336', 'Attack')]:
        vals = np.log1p(df.loc[df['label']==label, feat].clip(lower=0))
        ax.hist(vals, bins=50, alpha=0.5, color=color, label=name, density=True)
    ax.set_title(f'log1p({feat})')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../visualizations/02_feature_distributions.png', dpi=150)
plt.show()

In [ ]:
# ── Descriptive statistics by class ───────────────────────────────────────────
df.groupby('label')[['dur','sbytes','dbytes','Sload','Spkts','Sjit']].describe().T

---
## 4. Preprocessing

### ✅ Leak-free order
1. Encode categoricals  
2. **Split first** (train 80% / test 20%, stratified)  
3. Fit outlier caps on **training data only**, apply to both sets  
4. `StandardScaler` is inside the sklearn `Pipeline` — fits only on training data

In [ ]:
# Step 1 — Encode categoricals
le = LabelEncoder()
df_proc = df.copy()
for col in CATEGORICAL_FEATURES:
    if col in df_proc.columns:
        df_proc[col] = le.fit_transform(df_proc[col].astype(str))

features = [f for f in NUMERIC_FEATURES + CATEGORICAL_FEATURES if f in df_proc.columns]
X = df_proc[features].copy()
y = df_proc['label'].astype(int)

# Step 2 — Split FIRST (no leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train : {X_train.shape}  |  Test : {X_test.shape}')
print(f'Train class balance: {y_train.value_counts().to_dict()}')
print(f'Test  class balance: {y_test.value_counts().to_dict()}')

# Step 3 — Outlier clipping on training data only
caps = {}
for col in X_train.select_dtypes(include=[np.number]).columns:
    caps[col] = X_train[col].quantile(0.995)   # fit on train only
    X_train[col] = X_train[col].clip(upper=caps[col])
    X_test[col]  = X_test[col].clip(upper=caps[col])  # apply same cap to test

print(f'\nOutlier caps computed from training set only ✓')

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────────
corr = X_train.corr().abs()
top20 = corr.nlargest(20, corr.columns[0]).iloc[:, :20]

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(top20, annot=False, cmap='coolwarm', ax=ax, linewidths=0.3)
ax.set_title('Feature Correlation Matrix — Training Set (top 20)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../visualizations/03_correlation_heatmap.png', dpi=150)
plt.show()

---
## 5. Model Training with 5-Fold Cross-Validation

In [ ]:
models = {
    'RandomForest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    RandomForestClassifier(
            n_estimators=200, max_depth=20, min_samples_split=5,
            n_jobs=-1, random_state=42, class_weight='balanced'
        ))
    ]),
    'XGBoost': Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    XGBClassifier(
            n_estimators=200, max_depth=8, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8,
            use_label_encoder=False, eval_metric='logloss',
            random_state=42, n_jobs=-1
        ))
    ]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    print(f'\n► Training {name} …')

    # 5-fold CV on training set
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1', n_jobs=-1)
    print(f'  CV F1 : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}  (per fold: {np.round(cv_scores, 4)})')

    # Final fit on full training set
    model.fit(X_train, y_train)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        'Accuracy':    accuracy_score(y_test,  y_pred),
        'Precision':   precision_score(y_test, y_pred),
        'Recall':      recall_score(y_test,    y_pred),
        'F1-Score':    f1_score(y_test,        y_pred),
        'AUC-ROC':     roc_auc_score(y_test,   y_proba),
        'CV_F1_mean':  cv_scores.mean(),
        'CV_F1_std':   cv_scores.std(),
    }
    results[name] = {'model': model, 'metrics': metrics,
                     'y_pred': y_pred, 'y_proba': y_proba}

    print(f'  Accuracy  : {metrics["Accuracy"]:.4f}')
    print(f'  Precision : {metrics["Precision"]:.4f}')
    print(f'  Recall    : {metrics["Recall"]:.4f}')
    print(f'  F1-Score  : {metrics["F1-Score"]:.4f}')
    print(f'  AUC-ROC   : {metrics["AUC-ROC"]:.4f}')

    joblib.dump(model, f'../models/{name.lower()}_ids.pkl')
    print(f'  Saved → ../models/{name.lower()}_ids.pkl')

---
## 6. Evaluation & Metrics

In [ ]:
for name, res in results.items():
    print(f'\n{'='*55}')
    print(f'  {name} — Classification Report')
    print('='*55)
    print(classification_report(y_test, res['y_pred'],
                                target_names=['Normal','Attack']))

In [ ]:
# Summary table
metric_names = ['Accuracy','Precision','Recall','F1-Score','AUC-ROC','CV_F1_mean','CV_F1_std']
summary = pd.DataFrame([
    {'Model': name, **{m: res['metrics'][m] for m in metric_names}}
    for name, res in results.items()
]).set_index('Model')
summary.round(4)

---
## 7. Visualisations

In [ ]:
# ── Confusion matrices ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Normal','Attack'],
                yticklabels=['Normal','Attack'])
    ax.set_title(f'{name} — Confusion Matrix', fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('../visualizations/04_confusion_matrices.png', dpi=150)
plt.show()

In [ ]:
# ── ROC curves ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#1976D2', '#D32F2F']
for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    auc = res['metrics']['AUC-ROC']
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {auc:.4f})')
ax.plot([0,1],[0,1],'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Intrusion Detection', fontweight='bold')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../visualizations/05_roc_curves.png', dpi=150)
plt.show()

In [ ]:
# ── Precision-Recall curves ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
for (name, res), color in zip(results.items(), colors):
    prec, rec, _ = precision_recall_curve(y_test, res['y_proba'])
    ap = average_precision_score(y_test, res['y_proba'])
    ax.plot(rec, prec, color=color, lw=2, label=f'{name} (AP = {ap:.4f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../visualizations/06_precision_recall.png', dpi=150)
plt.show()

In [ ]:
# ── Feature importance ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, (name, res) in zip(axes, results.items()):
    clf = res['model'].named_steps['clf']
    imp = clf.feature_importances_
    idx = np.argsort(imp)[-20:]
    ax.barh([features[i] for i in idx], imp[idx], color='#5C6BC0')
    ax.set_title(f'{name} — Top 20 Features', fontweight='bold')
    ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('../visualizations/07_feature_importance.png', dpi=150)
plt.show()

In [ ]:
# ── Metrics comparison bar chart ──────────────────────────────────────────────
metric_names_plot = ['Accuracy','Precision','Recall','F1-Score','AUC-ROC']
x = np.arange(len(metric_names_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
for i, (name, res) in enumerate(results.items()):
    vals = [res['metrics'][m] for m in metric_names_plot]
    bars = ax.bar(x + i * width, vals, width, label=name,
                  color=colors[i], alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + width/2)
ax.set_xticklabels(metric_names_plot)
ax.set_ylim(0.85, 1.02)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../visualizations/08_metrics_comparison.png', dpi=150)
plt.show()

---
## 8. Save Artefacts

In [ ]:
# Metrics CSV
summary.reset_index().to_csv('../results/model_metrics.csv', index=False)
print('Saved → ../results/model_metrics.csv')

# Classification reports
with open('../results/classification_report.txt', 'w') as f:
    for name, res in results.items():
        f.write(f'{'='*55}\n{name}\n{'='*55}\n')
        f.write(classification_report(y_test, res['y_pred'],
                                      target_names=['Normal','Attack']))
        f.write('\n')
print('Saved → ../results/classification_report.txt')
print('\n✅ All artefacts saved. Notebook complete.')

---
## References

1. Moustafa, N., & Slay, J. (2015). UNSW-NB15: A comprehensive data set for network intrusion detection systems. *MilCIS 2015*. https://doi.org/10.1109/MilCIS.2015.7348942  
2. Breiman, L. (2001). Random forests. *Machine Learning*, 45(1), 5–32.  
3. Chen, T., & Guestrin, C. (2016). XGBoost: A scalable tree boosting system. *KDD 2016*.  